In [66]:
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.dataframe
import pyspark.sql.functions as f
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
import math as m
from pyspark.ml.stat import Correlation
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.window import Window
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
from functools import reduce
from pyspark.ml.evaluation import RegressionEvaluator
import random

In [2]:
spark = (
    SparkSession.builder.appName("Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/08 12:25:16 WARN Utils: Your hostname, Lachys-Laptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/10/08 12:25:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/08 12:25:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.sparkContext.setLogLevel("ERROR")
import logging
logging.getLogger('org.apache.spark.sql.execution.window.WindowExec').setLevel(logging.ERROR)

In [81]:
def create_pl(lags, time_length, type):

    categorical_cols = ["biz_tags","rev_band"]
    indexer = StringIndexer(inputCols=categorical_cols, outputCols=[f"{c}_idx" for c in categorical_cols], handleInvalid="keep")
    encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in categorical_cols],
                        outputCols=[f"{c}_vec" for c in categorical_cols])
    
    feature_cols = (
        ["revenue"] +
        [f"rev_lag{l}" for l in lags] +
        [f"rev_mean_{w}" for w in [3,6,12]] +
        [f"rev_std_{w}" for w in [3,6,12]] +
        ["sin_month", "cos_month"] +
        [f"{c}_vec" for c in categorical_cols]
        )

    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    gbt = GBTRegressor(labelCol=f"{type}_{time_length}m", featuresCol="features", maxIter=100)   
    pipeline = Pipeline(stages=[indexer, encoder, assembler, gbt])
    return pipeline

def find_NULL(dfs):
    for df in dfs:
        condition = f.lit(False)
        
        condition = condition | f.col('rev_future_6m').isNull() & f.col('rev_future_12m').isNotNull()

        df.filter(condition).show()
    return df.filter(condition).show()

def impute_rev_lags_by_business(data, lags, group_col, order_col='year_month'):
    """
    Imputes missing revenue lag columns (rev_lag1, rev_lag3, etc.)
    with the mean of that lag for each business.
    """
    base_window = Window.partitionBy(group_col).orderBy(order_col) \
                        .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

    for l in lags:
        colname = f"rev_lag{l}"
        first_col = f"{colname}_first"
        
        # compute business-specific mean and fill nulls
        data = (
            data
            .withColumn(first_col, f.first(col(colname), ignorenulls=True).over(base_window))
            .withColumn(
                colname,
                f.when(col(colname).isNull(), col(first_col)).otherwise(col(colname))
            )
            .drop(first_col)
        )

    return data

def create_model(data, time_length, type):

    lags=[1,3,6,12]
    required_cols = [f"{type}_{time_length}m"] #+ [f"rev_lag{l}" for l in lags]
    data=data.na.drop(subset=required_cols)
    data=impute_rev_lags_by_business(data, lags, 'merchant_abn', 'year_month')
    data=data.na.fill(0)
    pipeline = create_pl(lags, time_length, type)
    train, test = data.randomSplit([0.8,0.2], seed=42)
    model = pipeline.fit(train)
    predictions = model.transform(test)
    
    return predictions
    
def performance(predictions, time_length, type):
    
    rmse_rev = RegressionEvaluator(
        labelCol=f"{type}_{time_length}m",
        predictionCol="prediction",
        metricName="rmse"
    )

    mae_rev = RegressionEvaluator(
        labelCol=f"{type}_{time_length}m",
        predictionCol="prediction",
        metricName="mae"
    )


    rmse = rmse_rev.evaluate(predictions)
    mae = mae_rev.evaluate(predictions)

    
    ranked = (
        predictions.groupBy("merchant_abn")
        .agg(f.avg("prediction").alias(f"pred_{type}"))
        
    )

    window = Window.partitionBy(f.lit(1)).orderBy(f.desc(f"pred_{type}"))
    
    # Add ranking column
    ranked = ranked.withColumn(f"{type}_rank", f.row_number().over(window))
    #print(f"RMSE: {rmse:.4f}")
    return mae, rmse, ranked

def combine(growth, revenue):
    combined = growth.join(revenue,
        on="merchant_abn",
        how="inner"  # or "outer" if some merchants are missing in one DF
        )

    # Create a composite rank (sum of ranks, lower is better)
    combined = (combined.withColumn(
            "composite_rank",
            f.round(0.2*f.col("growth_rank") + 0.8*f.col("rev_future_rank"),4)
            )
        .orderBy('composite_rank', ascending=True)
        )

    # Optionally, sort by the composite rank
    combined = combined.orderBy(f.asc("composite_rank"))
    return combined
    

In [18]:
merchant_transactions=spark.read.parquet('../data/curated/merchant_transactions')

In [19]:
merchant_abn_name=merchant_transactions.groupBy('merchant_abn', 'business').count()
merchant_abn_name

merchant_abn,business,count
93693693468,Ipsum Dolor Corpo...,9
17739089622,Auctor Quis Corp.,17945
95279812400,Eu Dui Cum Ltd,7235
19237425345,A Scelerisque Ass...,12273
10142254217,Arcu Ac Orci Corp...,3036
43660707274,Nulla Integer Vul...,5587
71961434094,Amet Diam Corpora...,29665
80132164373,Integer Sem Corpo...,163
24830778398,Fames Ac Turpis Ltd,692
40252040480,Luctus Felis Puru...,5188


In [20]:
merchant_transactions=merchant_transactions.withColumn("year_month", f.date_format("order_datetime", "yyyy-MM"))
merchant_transactions=merchant_transactions.drop('user_id', 'business', 'order_datetime')

In [21]:
merchant_transactions

merchant_abn,dollar_value,biz_tags,rev_band,take_rate,year_month
67609108741,86.4040605836911,"cable, satellite,...",e,0.38,2021-08
98416475066,117.73979061019924,"cable, satellite,...",a,6.88,2022-03
47663262928,36.69873283148887,"cable, satellite,...",a,6.66,2021-08
69703285964,11.370554161545108,"cable, satellite,...",a,5.77,2022-03
15299889494,69.00314523230432,"cable, satellite,...",b,5.01,2021-08
80508375382,84.69498893548379,"cable, satellite,...",a,5.56,2022-03
10142254217,70.72395057645588,"cable, satellite,...",b,4.22,2021-08
21439773999,322.2119596790566,"cable, satellite,...",a,6.10,2022-03
29521780474,68.74975650290615,"cable, satellite,...",a,5.93,2021-08
85139489422,31.23106909347325,"cable, satellite,...",b,4.25,2022-03


In [22]:
month_agg=(merchant_transactions.groupBy('merchant_abn','year_month', 'biz_tags', 'rev_band')
                                .agg(f.round(f.mean('take_rate'),4).alias('ave_take_rate'),
                                     f.round(f.sum('dollar_value'),4).alias('revenue'),
                                     f.round(col('revenue')*col('ave_take_rate'),4).alias('taking'))
)

In [23]:
month_agg=month_agg.orderBy('merchant_abn', 'year_month')
month_agg

merchant_abn,year_month,biz_tags,rev_band,ave_take_rate,revenue,taking
10023283211,2021-02,"furniture, home f...",e,0.18,701.5666,126.282
10023283211,2021-03,"furniture, home f...",e,0.18,24634.3455,4434.1822
10023283211,2021-04,"furniture, home f...",e,0.18,27622.3428,4972.0217
10023283211,2021-05,"furniture, home f...",e,0.18,30111.8833,5420.139
10023283211,2021-06,"furniture, home f...",e,0.18,28790.4343,5182.2782
10023283211,2021-07,"furniture, home f...",e,0.18,29430.7319,5297.5317
10023283211,2021-08,"furniture, home f...",e,0.18,32118.4915,5781.3285
10023283211,2021-09,"furniture, home f...",e,0.18,34755.7764,6256.0398
10023283211,2021-10,"furniture, home f...",e,0.18,39993.2919,7198.7925
10023283211,2021-11,"furniture, home f...",e,0.18,47544.3817,8557.9887


In [25]:
taking=month_agg.groupBy('merchant_abn').agg(
                    f.sum('taking').alias('total_taking')
)

window = Window.partitionBy(f.lit(1)).orderBy(f.desc('total_taking'))
taking=taking.withColumn('rank', f.row_number().over(window))
taking

merchant_abn,total_taking,rank
32361057556,6.15238464441E7,1
86578477987,6.13410604311E7,2
45629217853,5.8569821692600004E7,3
96680767841,5.7745259905300006E7,4
21439773999,5.7364456265800014E7,5
64403598239,5.599137161039999E7,6
82368304209,5.4058938614999995E7,7
89726005175,5.329136659319999E7,8
94493496784,5.1109337113E7,9
49322182190,4.987914906429999E7,10


In [27]:
window = Window.partitionBy("merchant_abn").orderBy("year_month")
lags = [3,6,12]
new_data=month_agg.withColumn(f"rev_lag1", f.lag("revenue", 1).over(window))
for lag in lags:
    new_data = new_data.withColumn(f"rev_lag{lag}", f.lag("revenue", lag).over(window))

new_data

merchant_abn,year_month,biz_tags,rev_band,ave_take_rate,revenue,taking,rev_lag1,rev_lag3,rev_lag6,rev_lag12
10023283211,2021-02,"furniture, home f...",e,0.18,701.5666,126.282,NULL,NULL,NULL,NULL
10023283211,2021-03,"furniture, home f...",e,0.18,24634.3455,4434.1822,701.5666,NULL,NULL,NULL
10023283211,2021-04,"furniture, home f...",e,0.18,27622.3428,4972.0217,24634.3455,NULL,NULL,NULL
10023283211,2021-05,"furniture, home f...",e,0.18,30111.8833,5420.139,27622.3428,701.5666,NULL,NULL
10023283211,2021-06,"furniture, home f...",e,0.18,28790.4343,5182.2782,30111.8833,24634.3455,NULL,NULL
10023283211,2021-07,"furniture, home f...",e,0.18,29430.7319,5297.5317,28790.4343,27622.3428,NULL,NULL
10023283211,2021-08,"furniture, home f...",e,0.18,32118.4915,5781.3285,29430.7319,30111.8833,701.5666,NULL
10023283211,2021-09,"furniture, home f...",e,0.18,34755.7764,6256.0398,32118.4915,28790.4343,24634.3455,NULL
10023283211,2021-10,"furniture, home f...",e,0.18,39993.2919,7198.7925,34755.7764,29430.7319,27622.3428,NULL
10023283211,2021-11,"furniture, home f...",e,0.18,47544.3817,8557.9887,39993.2919,32118.4915,30111.8833,NULL


In [28]:
for window_size in [3,6,12]:
    roll_window = Window.partitionBy("merchant_abn").orderBy("year_month").rowsBetween(-window_size+1,0)
    new_data = (
        new_data
        .withColumn(f"rev_mean_{window_size}", f.mean("revenue").over(roll_window))
        .withColumn(f"rev_std_{window_size}", f.stddev("revenue").over(roll_window))
    )

new_data

merchant_abn,year_month,biz_tags,rev_band,ave_take_rate,revenue,taking,rev_lag1,rev_lag3,rev_lag6,rev_lag12,rev_mean_3,rev_std_3,rev_mean_6,rev_std_6,rev_mean_12,rev_std_12
10023283211,2021-02,"furniture, home f...",e,0.18,701.5666,126.282,NULL,NULL,NULL,NULL,701.5666,NULL,701.5666,NULL,701.5666,NULL
10023283211,2021-03,"furniture, home f...",e,0.18,24634.3455,4434.1822,701.5666,NULL,NULL,NULL,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325
10023283211,2021-04,"furniture, home f...",e,0.18,27622.3428,4972.0217,24634.3455,NULL,NULL,NULL,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884
10023283211,2021-05,"furniture, home f...",e,0.18,30111.8833,5420.139,27622.3428,701.5666,NULL,NULL,27456.19053333333,2742.5462656801387,20767.53455,13563.437941753089,20767.53455,13563.437941753089
10023283211,2021-06,"furniture, home f...",e,0.18,28790.4343,5182.2782,30111.8833,24634.3455,NULL,NULL,28841.553466666664,1245.557245647741,22372.114500000003,12282.040574739623,22372.114500000003,12282.040574739623
10023283211,2021-07,"furniture, home f...",e,0.18,29430.7319,5297.5317,28790.4343,27622.3428,NULL,NULL,29444.34983333333,660.8297443225855,23548.550733333337,11357.060790989783,23548.550733333337,11357.060790989783
10023283211,2021-08,"furniture, home f...",e,0.18,32118.4915,5781.3285,29430.7319,30111.8833,701.5666,NULL,30113.219233333333,1765.8802059421507,28784.704883333336,2524.657921826413,24772.827985714288,10861.752853173075
10023283211,2021-09,"furniture, home f...",e,0.18,34755.7764,6256.0398,32118.4915,28790.4343,24634.3455,NULL,32101.6666,2662.5621194049304,30471.61003333333,2577.743441343612,26020.696537500004,10657.444761577193
10023283211,2021-10,"furniture, home f...",e,0.18,39993.2919,7198.7925,34755.7764,29430.7319,27622.3428,NULL,35622.51993333334,4008.310566733992,32533.434883333335,4248.79224668721,27573.207133333333,11003.458096974942
10023283211,2021-11,"furniture, home f...",e,0.18,47544.3817,8557.9887,39993.2919,32118.4915,30111.8833,NULL,40764.48333333333,6429.0869141684525,35438.85128333333,7198.293176792175,29570.324589999997,12145.286021745746


In [29]:
new_data = (
    new_data
    .withColumn("rev_future_1m", f.lead("revenue", 1).over(window))
    .withColumn("growth_1m", (f.col("rev_future_1m") - f.col("revenue")) / f.col("revenue"))
    .withColumn("rev_future_3m", f.lead("revenue", 3).over(window))
    .withColumn("growth_3m", (f.col("rev_future_3m") - f.col("revenue")) / f.col("revenue"))
    .withColumn("rev_future_6m", f.lead("revenue", 6).over(window))
    .withColumn("growth_6m", (f.col("rev_future_6m") - f.col("revenue")) / f.col("revenue"))
    .withColumn("rev_future_12m", f.lead("revenue", 12).over(window))
    .withColumn("growth_12m", (f.col("rev_future_12m") - f.col("revenue")) / f.col("revenue"))
)

new_data

merchant_abn,year_month,biz_tags,rev_band,ave_take_rate,revenue,taking,rev_lag1,rev_lag3,rev_lag6,rev_lag12,rev_mean_3,rev_std_3,rev_mean_6,rev_std_6,rev_mean_12,rev_std_12,rev_future_1m,growth_1m,rev_future_3m,growth_3m,rev_future_6m,growth_6m,rev_future_12m,growth_12m
10023283211,2021-02,"furniture, home f...",e,0.18,701.5666,126.282,NULL,NULL,NULL,NULL,701.5666,NULL,701.5666,NULL,701.5666,NULL,24634.3455,34.113338491313584,30111.8833,41.92091912585349,32118.4915,44.78110118127061,30182.6491,42.021787382694676
10023283211,2021-03,"furniture, home f...",e,0.18,24634.3455,4434.1822,701.5666,NULL,NULL,NULL,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325,27622.3428,0.12129395928136183,28790.4343,0.16871115167236742,34755.7764,0.41086664551327345,31514.377,0.27928614949400626
10023283211,2021-04,"furniture, home f...",e,0.18,27622.3428,4972.0217,24634.3455,NULL,NULL,NULL,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884,30111.8833,0.0901277823545077,29430.7319,0.06546834615346242,39993.2919,0.4478602408771786,36365.1443,0.3165119469880738
10023283211,2021-05,"furniture, home f...",e,0.18,30111.8833,5420.139,27622.3428,701.5666,NULL,NULL,27456.19053333333,2742.5462656801387,20767.53455,13563.437941753089,20767.53455,13563.437941753089,28790.4343,-0.04388463474152...,32118.4915,0.06663841580443422,47544.3817,0.578924214946064,35454.8418,0.17743687589278084
10023283211,2021-06,"furniture, home f...",e,0.18,28790.4343,5182.2782,30111.8833,24634.3455,NULL,NULL,28841.553466666664,1245.557245647741,22372.114500000003,12282.040574739623,22372.114500000003,12282.040574739623,29430.7319,0.022239942382529396,34755.7764,0.20719875351098824,44094.0579,0.5315523705038377,35735.9922,0.24124533265550632
10023283211,2021-07,"furniture, home f...",e,0.18,29430.7319,5297.5317,28790.4343,27622.3428,NULL,NULL,29444.34983333333,660.8297443225855,23548.550733333337,11357.060790989783,23548.550733333337,11357.060790989783,32118.4915,0.09132493235752663,39993.2919,0.35889559375857716,29431.268,1.821565300593731E-5,40122.3353,0.36328024176660045
10023283211,2021-08,"furniture, home f...",e,0.18,32118.4915,5781.3285,29430.7319,30111.8833,701.5666,NULL,30113.219233333333,1765.8802059421507,28784.704883333336,2524.657921826413,24772.827985714288,10861.752853173075,34755.7764,0.0821111072417583,47544.3817,0.4802806570165351,30182.6491,-0.06027189664246...,36497.3556,0.1363346749955552
10023283211,2021-09,"furniture, home f...",e,0.18,34755.7764,6256.0398,32118.4915,28790.4343,24634.3455,NULL,32101.6666,2662.5621194049304,30471.61003333333,2577.743441343612,26020.696537500004,10657.444761577193,39993.2919,0.15069482090464806,44094.0579,0.26868286274278125,31514.377,-0.09326217785196712,44448.763,0.27888850729284803
10023283211,2021-10,"furniture, home f...",e,0.18,39993.2919,7198.7925,34755.7764,29430.7319,27622.3428,NULL,35622.51993333334,4008.310566733992,32533.434883333335,4248.79224668721,27573.207133333333,11003.458096974942,47544.3817,0.1888089087260157,29431.268,-0.2640948868727657,36365.1443,-0.09071890378696226,36755.0168,-0.08097045644797245
10023283211,2021-11,"furniture, home f...",e,0.18,47544.3817,8557.9887,39993.2919,32118.4915,30111.8833,NULL,40764.48333333333,6429.0869141684525,35438.85128333333,7198.293176792175,29570.324589999997,12145.286021745746,44094.0579,-0.07257058934473426,30182.6491,-0.36516896380208896,35454.8418,-0.25427904344794533,NULL,NULL


In [31]:
new_data = (
    new_data
    .withColumn("month", f.month("year_month"))
    .withColumn("sin_month", f.sin(2 * m.pi * col("month") / 12))
    .withColumn("cos_month", f.cos(2 * m.pi * col("month") / 12))
)

new_data

merchant_abn,year_month,biz_tags,rev_band,ave_take_rate,revenue,taking,rev_lag1,rev_lag3,rev_lag6,rev_lag12,rev_mean_3,rev_std_3,rev_mean_6,rev_std_6,rev_mean_12,rev_std_12,rev_future_1m,growth_1m,rev_future_3m,growth_3m,rev_future_6m,growth_6m,rev_future_12m,growth_12m,month,sin_month,cos_month
10023283211,2021-02,"furniture, home f...",e,0.18,701.5666,126.282,NULL,NULL,NULL,NULL,701.5666,NULL,701.5666,NULL,701.5666,NULL,24634.3455,34.113338491313584,30111.8833,41.92091912585349,32118.4915,44.78110118127061,30182.6491,42.021787382694676,2,0.8660254037844386,0.5000000000000001
10023283211,2021-03,"furniture, home f...",e,0.18,24634.3455,4434.1822,701.5666,NULL,NULL,NULL,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325,27622.3428,0.12129395928136183,28790.4343,0.16871115167236742,34755.7764,0.41086664551327345,31514.377,0.27928614949400626,3,1.0,6.123233995736766...
10023283211,2021-04,"furniture, home f...",e,0.18,27622.3428,4972.0217,24634.3455,NULL,NULL,NULL,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884,30111.8833,0.0901277823545077,29430.7319,0.06546834615346242,39993.2919,0.4478602408771786,36365.1443,0.3165119469880738,4,0.8660254037844387,-0.4999999999999998
10023283211,2021-05,"furniture, home f...",e,0.18,30111.8833,5420.139,27622.3428,701.5666,NULL,NULL,27456.19053333333,2742.5462656801387,20767.53455,13563.437941753089,20767.53455,13563.437941753089,28790.4343,-0.04388463474152...,32118.4915,0.06663841580443422,47544.3817,0.578924214946064,35454.8418,0.17743687589278084,5,0.49999999999999994,-0.8660254037844387
10023283211,2021-06,"furniture, home f...",e,0.18,28790.4343,5182.2782,30111.8833,24634.3455,NULL,NULL,28841.553466666664,1245.557245647741,22372.114500000003,12282.040574739623,22372.114500000003,12282.040574739623,29430.7319,0.022239942382529396,34755.7764,0.20719875351098824,44094.0579,0.5315523705038377,35735.9922,0.24124533265550632,6,1.224646799147353...,-1.0
10023283211,2021-07,"furniture, home f...",e,0.18,29430.7319,5297.5317,28790.4343,27622.3428,NULL,NULL,29444.34983333333,660.8297443225855,23548.550733333337,11357.060790989783,23548.550733333337,11357.060790989783,32118.4915,0.09132493235752663,39993.2919,0.35889559375857716,29431.268,1.821565300593731E-5,40122.3353,0.36328024176660045,7,-0.4999999999999997,-0.8660254037844388
10023283211,2021-08,"furniture, home f...",e,0.18,32118.4915,5781.3285,29430.7319,30111.8833,701.5666,NULL,30113.219233333333,1765.8802059421507,28784.704883333336,2524.657921826413,24772.827985714288,10861.752853173075,34755.7764,0.0821111072417583,47544.3817,0.4802806570165351,30182.6491,-0.06027189664246...,36497.3556,0.1363346749955552,8,-0.8660254037844384,-0.5000000000000004
10023283211,2021-09,"furniture, home f...",e,0.18,34755.7764,6256.0398,32118.4915,28790.4343,24634.3455,NULL,32101.6666,2662.5621194049304,30471.61003333333,2577.743441343612,26020.696537500004,10657.444761577193,39993.2919,0.15069482090464806,44094.0579,0.26868286274278125,31514.377,-0.09326217785196712,44448.763,0.27888850729284803,9,-1.0,-1.83697019872102...
10023283211,2021-10,"furniture, home f...",e,0.18,39993.2919,7198.7925,34755.7764,29430.7319,27622.3428,NULL,35622.51993333334,4008.310566733992,32533.434883333335,4248.79224668721,27573.207133333333,11003.458096974942,47544.3817,0.1888089087260157,29431.268,-0.2640948868727657,36365.1443,-0.09071890378696226,36755.0168,-0.08097045644797245,10,-0.8660254037844386,0.5000000000000001
10023283211,2021-11,"furniture, home f...",e,0.18,47544.3817,8557.9887,39993.2919,32118.4915,30111.8833,NULL,40764.48333333333,6429.0869141684525,35438.85128333333,7198.293176792175,29570.324589999997,12145.286021745746,44094.0579,-0.07257058934473426,30182.6491,-0.36516896380208896,35454.8418,-0.25427904344794533,NULL,NULL,11,-0.5000000000000004,0.8660254037844384


In [86]:
predictions_growth_1 = create_model(new_data, 1, 'growth')

In [82]:
mae_g_1, rmse_g_1, ranked_growth_1 = performance(predictions_growth_1, 1, 'growth')
mae_g_1, rmse_g_1

(1.4677183740281754, 15.866914349018893)

In [83]:
predictions_rev_1=create_model(new_data, 1, 'rev_future')

In [85]:
mae_r_1, rsme_r_1, ranked_rev_1 = performance(predictions_rev_1, 1, 'rev_future')
mae_r_1, rsme_r_1


(3828.535400610306, 12944.002216722878)

In [47]:
composite_rank_1m=combine(ranked_growth_1, ranked_rev_1)
composite_rank_1m

merchant_abn,pred_growth,growth_rank,pred_rev_future,rev_future_rank,composite_rank
35909341340,9.039592858429321,146,486001.1081406626,2,30.800000000000004
32709545238,13.289719076922541,85,400449.82600236684,21,33.8
39649557865,6.9436473198606254,218,440265.6798774948,9,50.800000000000004
35223308778,27.415047380383264,23,303116.18259330397,58,51.00000000000001
52959528548,13.406656300852118,83,263144.92785482487,70,72.6
28057731482,5.33414880282829,317,415250.7412791204,15,75.4
45629217853,5.54801955964835,299,396238.14523332147,24,79.0
68216911708,6.817888314041436,224,338253.62730997993,45,80.80000000000001
22033359776,5.322591618586747,319,371983.436426838,35,91.80000000000001
72472909171,7.1179722598686,208,286461.5126206106,66,94.4


In [49]:
predictions_rev_3 = create_model(new_data, 3,'rev_future')

In [50]:
rmse_r_3, ranked_rev_3 = performance(predictions_rev_3, 3, 'rev_future')
rmse_r_3

'RMSE: 12596.7490'

In [51]:
predictions_growth_3 = create_model(new_data, 3, 'growth')

In [52]:
rmse_g_3, ranked_growth_3 = performance(predictions_growth_3, 3, 'growth')
rmse_g_3

'RMSE: 30.4425'

In [53]:
composite_rank_3m=combine(ranked_growth_3, ranked_rev_3)
composite_rank_3m

merchant_abn,pred_growth,growth_rank,pred_rev_future,rev_future_rank,composite_rank
80324045558,16.563463995736065,90,345509.0285293588,47,55.6
94493496784,7.989439648769867,268,467383.77834008826,4,56.8
45629217853,8.319989520039469,259,428310.1842238207,16,64.6
35909341340,6.756975270805458,331,474840.15826127527,2,67.8
76767266140,7.697096206289424,280,427292.63621909235,17,69.6
52959528548,6.865967870268013,327,450743.098347744,10,73.4
22033359776,7.73552570657363,277,386017.8002000664,29,78.6
43186523025,5.244876651412539,406,462633.01310771564,5,85.2
11439466003,12.437405227003529,147,237485.43113757888,71,86.2
64403598239,7.445166557406453,302,362813.2120040442,42,94.0


In [54]:
predictions_rev_6 = create_model(new_data, 6, 'rev_future')

In [55]:
rmse_r_6, ranked_rev_6=performance(predictions_rev_6, 6, 'rev_future')
rmse_r_6

'RMSE: 11708.0177'

In [56]:
predictions_growth_6 = create_model(new_data, 6, 'growth')

In [57]:
rmse_g_6, ranked_growth_6 = performance(predictions_growth_6, 6, 'growth')
rmse_g_6

'RMSE: 36.2908'

In [58]:
composite_rank_6m=combine(ranked_growth_6, ranked_rev_6)
composite_rank_6m

merchant_abn,pred_growth,growth_rank,pred_rev_future,rev_future_rank,composite_rank
43186523025,32.67648493137961,40,376219.4825022558,42,41.6
49891706470,18.870735352877787,101,378377.8961756293,40,52.2
35223308778,17.78222012836475,113,372796.86235613876,43,57.0
63123845164,17.8244109677135,112,316599.9682027581,60,70.4
80518954462,11.682109552169957,210,380021.2789605991,39,73.2
49322182190,8.692298393831791,305,437452.0095024819,18,75.4
93558142492,11.77097896306604,206,361864.66591211833,47,78.8
96680767841,7.291370976480174,354,424859.7894494634,22,88.4
13514558491,13.087461343603733,186,288819.3109645774,65,89.2
37379915451,8.093675843088901,330,391752.2896825529,34,93.2


In [59]:
predictions_rev_12 = create_model(new_data, 12, 'rev_future')

In [60]:
rmse_r_12, ranked_rev_12 = performance(predictions_rev_12, 12, 'rev_future')
rmse_r_12

'RMSE: 14430.2593'

In [61]:
predictions_growth_12 = create_model(new_data, 12, 'growth')

In [62]:
rmse_g_12, ranked_growth_12 = performance(predictions_growth_12, 12, 'growth')
rmse_g_12

'RMSE: 66.7782'

In [63]:
composite_rank_12m=combine(ranked_growth_12, ranked_rev_12)
composite_rank_12m

merchant_abn,pred_growth,growth_rank,pred_rev_future,rev_future_rank,composite_rank
62692834922,23.494520289408054,105,310039.21341810835,56,65.8
60956456424,14.22120582942462,170,352065.1910515145,43,68.4
78760357380,29.74279707780812,63,184010.8019202134,76,73.4
67978471888,13.558895338523918,188,337716.19763309555,48,76.0
49505931725,15.268342917340428,154,288446.8213437245,58,77.2
39649557865,12.627228821550798,210,343328.37433753454,47,79.6
94690988633,12.807640539533221,204,323693.99756168027,51,81.6
49212265466,12.447682500510266,215,328238.4876005901,50,83.0
32361057556,7.033322176634575,358,424867.39984259725,18,86.0
45629217853,7.233860059745216,354,421444.9846331017,21,87.6


In [64]:
rank_1 = composite_rank_1m.withColumnRenamed("composite_rank", "rank_1").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank')
rank_3 = composite_rank_3m.withColumnRenamed("composite_rank", "rank_3").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank')
rank_6 = composite_rank_6m.withColumnRenamed("composite_rank", "rank_6").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank')
rank_12 = composite_rank_12m.withColumnRenamed("composite_rank", "rank_12").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank')

# Join on merchant_abn
combined = reduce(lambda a, b: a.join(b, on="merchant_abn", how="outer"), [rank_1,rank_3,rank_6,rank_12])

# Compute average of available (non-null) ranks
combined = combined.withColumn(
    "ave_rank",
    (
        f.coalesce(col("rank_1"), f.lit(0)) +
        f.coalesce(col("rank_3"), f.lit(0)) +
        f.coalesce(col("rank_6"), f.lit(0)) +
        f.coalesce(col("rank_12"), f.lit(0))
    ) / (
        f.when(col("rank_1").isNotNull(), 1).otherwise(0) +
        f.when(col("rank_3").isNotNull(), 1).otherwise(0) +
        f.when(col("rank_6").isNotNull(), 1).otherwise(0) +
        f.when(col("rank_12").isNotNull(), 1).otherwise(0)
    )
).orderBy('ave_rank', ascending=True)

combined

merchant_abn,rank_1,rank_3,rank_6,rank_12,ave_rank
24674067743,94.80000000000001,NULL,NULL,NULL,94.80000000000001
45629217853,79.0,64.6,386.2,87.6,154.35
35909341340,30.800000000000004,67.8,204.0,368.6,167.8
11439466003,121.2,86.2,136.2,394.8,184.60000000000002
29616684420,153.2,158.0,298.0,NULL,203.0666666666667
60829135130,169.4,121.2,435.2,116.6,210.6
22033359776,91.80000000000001,78.6,385.6,332.6,222.15
46804135891,503.40000000000003,103.0,221.2,104.8,233.10000000000002
37379915451,372.6,114.2,93.2,361.0,235.25
43186523025,370.6,85.2,41.6,456.0,238.35000000000002
